# PML_SUMO

This notebook performs the following analysis:
- Detect Nucleus in 3D or 2D
- Detect Foci in 3D or 2D
- Measure :
-          the volume of each Foci
-          the background-corrected mean and sum SUMO and PML intensities within each Foci
-          the background-corrected mean and sum SUMO and PML intensities outside Foci but inside each nucleus
  

**Authors:** Thomas Caille & Héloïse Monnet, ORION image analysis facility

**Source code and additional information:** <https://github.com/orion-cirb/PML_SUMO> (access restricted - requires GitHub account with granted permissions)

## 1. Import libraries

In [1]:


from aicsimageio import AICSImage
from cellpose import models
import datetime
import logging
import matplotlib.pyplot as plt
import napari
import numpy as np
import os
import pandas as pd
import scipy as sc
import skimage as sk
import tifffile as tf


from scipy.ndimage import convolve
from typing import List

## 2. Enter parameters

In [53]:
# Directory containing the input images for processing (must end with '/')
img_dir = 'C:/Users/utilisateur/Adele/Images/Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/'

# Directory name where results will be saved (must end with '/')
result_dir_name = 'results/'

# Channel indices in the image data (1-based indexing)
channel_DAPI = 3          # Channel index for Nucleus marker
channel_PML = 1           # Channel index for PML marker
channel_SUMO = 2          # Channel index for SUMO marker

# Parameters for the 3D detection of Nucleus with the 2D-stitched version of Cellpose 3
cellpose_voxel_size = 0.055
cellpose_model_dir = 'C:/Users/utilisateur/.cellpose/models/'
cellpose_model = 'nuclei'
cellpose_diameter = 204.8
cellpose_cellprob_threshold = 0
cellpose_flow_threshold = 0.4
cellpose_stitch_threshold = 0.5

# Volume constraints to filter detected Foci
foci_vol_min_um = 0.000002      # Minimum allowed cell volume (in µm³)
foci_vol_max_um = 0.8    # Maximum allowed cell volume (in µm³)



## 3. Declare utility functions

In [54]:
def open_img_czi(img_dir, img_name, verbose=False):
    '''
    Open a .czi image file and extract voxel size from metadata 

    Parameters:
        img_dir (str): Directory containing the .czi image
        img_name (str): Name of the image file

    Returns:
        img (ndarray): 4D image array (C, Z, Y, X)
        voxel_size (list of float): [Z, Y, X] voxel dimensions in µm
    '''
    czi = AICSImage(img_dir+img_name)
    img = czi.get_image_data("CZYX", T=0)
    voxel_size = [czi.physical_pixel_sizes.Z, czi.physical_pixel_sizes.Y, czi.physical_pixel_sizes.X]

    if verbose:
        print('Voxel size (ZYX):', voxel_size, 'µm')
        print('Image size (CZYX):', img.shape)

    return img, voxel_size

def get_napari_viewer():
    """
    Launch or retrieve the current Napari viewer in 3D mode.

    Returns:
        napari.Viewer: Active or newly created Napari viewer.
    """
    # Check if a viewer is already running
    viewer = napari.current_viewer()
    
    if viewer is None:
        # Create a new viewer
        viewer = napari.Viewer()
        napari.run()
    else:
        # Remove all existing layers
        viewer.layers.clear()

    #viewer.dims.ndisplay = 2
    return viewer

def cellpose_detection(img, model_dir, model, diameter, cellprob_threshold, flow_threshold, stitch_threshold):
    """
    Run 3D Cellpose segmentation on the input image.

    Parameters:
        img (ndarray): 3D image array (ZYX)
        model (str): Cellpose model name
        diameter (float): Estimated object diameter in pixels

    Returns:
        masks (ndarray): 3D label image with segmented objects
    """
    model = models.CellposeModel(gpu=True,pretrained_model=cellpose_model)
    masks, _, _ = model.eval(img,
                            diameter=diameter,
                            
                            cellprob_threshold=cellprob_threshold,
                            flow_threshold=flow_threshold,
                            
                            )
    return masks


def filter_labels_volume(label_img, min_voxels=0, max_voxels=np.inf):
    """
    Filter labeled regions by their volume (in voxels).

    Parameters:
        label_img (ndarray): 3D label image (ZYX)
        min_voxels (int): Minimum voxel count to keep a region
        max_voxels (int or float): Maximum voxel count to keep a region

    Returns:
        filtered (ndarray): 3D label image with size-filtered regions relabeled
    """
    valid_labels = [
        r.label for r in sk.measure.regionprops(label_img)
        if min_voxels <= r.area <= max_voxels
    ]
    filtered = np.isin(label_img, valid_labels) * label_img
    return sk.segmentation.relabel_sequential(filtered)[0]

def compute_background_per_slice(img):
    """
    Computes the mean background intensity per slice in a 3D image stack.
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        
    Returns:
        ndarray: Mean background intensity per slice
    """
    backgrounds = []
    
    for i in range(img.shape[0]):
        # Extract current slice
        img_bin = np.array(img[i])
         # Apply 2D Gaussian filter
        img_bin = sk.filters.gaussian(img_bin, sigma=10)                        
         # Apply Li thresholding
        thresh = sk.filters.threshold_li(img_bin)
        img_bin = img_bin > thresh
        # Invert binary mask to get background
        img_bin = sk.util.invert(img_bin)
        
        # Extract background pixels and compute mean intensity
        img_bg = np.array(img[i])
        img_bg = img_bg[np.nonzero(img_bin)]
        background = np.mean(img_bg) if len(img_bg) > 0 else 0
        
        backgrounds.append(background)
        
    return(np.array(backgrounds))

def MIP(img) :
    """
    Computes the Maximum intensity projection of a 3D image stack.
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        
    Returns:
        ndarray: 2D image (XY)
    """
    IM_MAX=np.max(img,axis=0)
    return (IM_MAX)


def laplacian_variance(img: np.ndarray) -> float:
    """
    Calculate the variance with a laplacian kernel.
    
    Parameters:
        img (ndarray): 2D image (XY)
        
    Returns:
        ndarray: variance of 2D image (XY) 
    """
    
    kernel = np.array([[0,  1, 0],
                       [1, -4, 1],
                       [0,  1, 0]])
    lap = convolve(img.astype(float), kernel)
    return np.var(lap)
    
Image = np.ndarray

def find_best_z_plane_id(img_list: List[Image],var_range) -> int:
    """
    Rank the variance slices 
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        float between 0 and 1 : variance coefficient
        
    Returns:
        table : all consecutive slices with variance in the range enter by the user
    """
    lap_vars_per_z_plane = []
    for img in img_list:
        lap_vars_per_z_plane.append(laplacian_variance(img))
    max_var = max(lap_vars_per_z_plane)
    index = np.where(lap_vars_per_z_plane > var_range*max_var)
    
    
    return index[0]

## 4. Load images

In [55]:
# List all files in the image directory with '.czi' file format
img_names = [item for item in os.listdir(img_dir) if item.endswith('czi')]
img_names.sort()
print(f"Found {len(img_names)} images to analyze in directory: {img_dir}")

# Create results folder
result_dir = img_dir + result_dir_name
os.makedirs(result_dir, exist_ok=True)
print(f"\nResults directory successfully created at: {result_dir}")

Found 12 images to analyze in directory: C:/Users/utilisateur/Adele/Images/Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/

Results directory successfully created at: C:/Users/utilisateur/Adele/Images/Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/results/


### Preview the image of your choice

In [9]:
# Open image

img_name = img_names[2]
print('Image name:', img_name)
img, voxel_size = open_img_czi(img_dir, img_name)
        
# Display image in Napari

viewer = get_napari_viewer()
viewer.add_image(img, name="image")

Image name: da2h pml 1 5000-3 PML_488 SUMO2_594.czi


<Image layer 'image' at 0x1cd700331f0>

## 5. Detect Nucleus

In [48]:
start_time = datetime.datetime.now()
logging.basicConfig(level=logging.WARN, force=True)

# Loop through each image file
for index, img_name in enumerate(img_names, start=1):
    print(f"\nAnalyzing image: {img_name} ({index}/{len(img_names)})")
    
    # Load image and voxel size from metadata
    img, voxel_size = open_img_czi(img_dir, img_name)
    
    # Extract ChaT/vChaT channel
    img_DAPI = img[channel_DAPI-1]

    # Perform a MIP on the DAPI channel to enhance cell detection
    img_DAPI_MIP = MIP(img_DAPI)
    
    # Run Cellpose detection
    nucleus = cellpose_detection(img_DAPI_MIP, cellpose_model_dir, cellpose_model, cellpose_diameter, cellpose_cellprob_threshold, cellpose_flow_threshold, cellpose_stitch_threshold)

    # Filter masks at XY border and by volume
    #cells_clean = filter_labels_xy_border(nucleus)
    nucleus_clean = filter_labels_volume(nucleus)
 
    print(f"Cells remaining after filtering: {np.max(nucleus_clean)}")
    
    # Save result as .tif file
    tf.imwrite(result_dir + img_name.replace(".czi", "_nucleus.tif"), nucleus_clean, imagej=True, resolution=(1/voxel_size[1], 1/voxel_size[2]), metadata={'spacing': voxel_size[0], 'unit': 'um'})
    
print(f"\nAnalysis done!")
print(f"Execution time: {datetime.datetime.now() - start_time}")


Analyzing image: C217Vn2 Mp2 1h DA-63x-1.czi (1/12)
Cells remaining after filtering: 19

Analyzing image: C217Vn2 Mp2 1h DA-63x-2.czi (2/12)
Cells remaining after filtering: 18

Analyzing image: C217Vn2 Mp2 2h DA-63x-1.czi (3/12)
Cells remaining after filtering: 23

Analyzing image: C217Vn2 Mp2 UT DA-63x-1.czi (4/12)
Cells remaining after filtering: 12

Analyzing image: E167R2 1h 63x-1.czi (5/12)
Cells remaining after filtering: 21

Analyzing image: E167R2 2h 63x-1.czi (6/12)
Cells remaining after filtering: 27

Analyzing image: E167R2 UT 63x-1.czi (7/12)
Cells remaining after filtering: 21

Analyzing image: E167R2 UT 63x-2.czi (8/12)
Cells remaining after filtering: 13

Analyzing image: WT 1h 63x-2.czi (9/12)
Cells remaining after filtering: 10

Analyzing image: WT 1h 63x-3.czi (10/12)
Cells remaining after filtering: 11

Analyzing image: WT 2h 63x-1.czi (11/12)
Cells remaining after filtering: 10

Analyzing image: WT UT 63x-1.czi (12/12)
Cells remaining after filtering: 9

Analysis 

### 6. Compute 3D PML foci detection and measure intensity 

In [56]:
start_time = datetime.datetime.now()
# declare utility variables
df_img_list = []
df_mask_list = []
df_bg_list = []
for index, img_name in enumerate(img_names, start=1):
    print(f"\nAnalyzing image: {img_name} ({index}/{len(img_names)})")
    img, voxel_size = open_img_czi(img_dir, img_name)  

    # Select the Dapi channel and find best Z planes
    img_DAPI = img[channel_DAPI-1]
    best_z = find_best_z_plane_id(img[channel_PML-1,:,:],0.6)
    print(best_z)
    # Open corresponding nucleus segmentation mask
    nucleus = sk.io.imread(result_dir + img_name.replace(".czi", "_nucleus.tif"))
    print(f"Cells number: {np.max(nucleus)}")

    # Extend the 2D mask from cell detection to the corresponding number of Z
    nb_stacks = img_DAPI.shape[0]
    shape = 1,img_DAPI.shape[1],img_DAPI.shape[2]
    nucleus= np.reshape(nucleus, newshape = shape)
    nucleus_focus = np.repeat(nucleus,nb_stacks,axis=0)
    # Keep nucleus segmentation on detected best z planes
    nucleus_focus[0:best_z[0],:,:] = 0
    nucleus_focus[best_z[-1]+1:nb_stacks,:,:]= 0
    
    mask = nucleus_focus > 0
    masked_nucleus = mask * nucleus_focus 
    # Calculate the voxel volume wich is the product between every pixel size
    voxel_vol = np.prod(voxel_size) 
 
    # Took only best z planes on PML and SUMO channel
    chanPML = img[channel_PML-1,:,:]
    chanSUMO = img[channel_SUMO-1,:,:]

    # PREPROCESSING
    # Apply Difference of Gaussian filter to enhance Foci detection
    chanPML_dog = sk.filters.difference_of_gaussians(chanPML,1,3)    
    chanPML_dog_int = sk.util.img_as_int(chanPML_dog)
    
    # THRESHOLDING
    # Threshold the PML channel preprocessed with the triangle method
    thr = sk.filters.threshold_triangle(chanPML_dog_int)
    thr_PML_chan = chanPML_dog_int > thr

    # POSTPROCESSING
    # Perform a 3D opening to remove noise and to separate touching foci
    thr_PML_chan = sk.morphology.binary_opening(thr_PML_chan,footprint= sk.morphology.ball(2))

    # Put a label on every objects detected and filter them by volume
    chanPML_labels = sk.morphology.label(thr_PML_chan, connectivity=1)
    filter_out = sk.measure.regionprops(chanPML_labels)
    labels_to_keep = [obj.label for obj in filter_out]
    filtered_label_image = sk.morphology.label(np.isin(chanPML_labels, labels_to_keep))
    filtered_label_image = filter_labels_volume(filtered_label_image,
                                        min_voxels=foci_vol_min_um / voxel_vol,
                                        max_voxels=foci_vol_max_um / voxel_vol)
    
    
    # Compute the PML and SUMO backgrounds for every best z slice and substract it to the original image
    backgrounds_PML = compute_background_per_slice(chanPML)
    img_PML_norm = np.array([chanPML[i] - backgrounds_PML[i] for i in range(chanPML.shape[0])])
    backgrounds_SUMO = compute_background_per_slice(chanSUMO)
    img_SUMO_norm = np.array([chanSUMO[i] - backgrounds_SUMO[i] for i in range(chanSUMO.shape[0])])

    tf.imwrite(result_dir + img_name.replace(".czi", "_foci_labels.tif"), filtered_label_image, imagej=False, resolution=(1/voxel_size[1], 1/voxel_size[2]), metadata={'spacing': voxel_size[0], 'unit': 'um'})

    # MEASUREMENTS
    # Measure objects intensity in normalized channels (PML and SUMO) 
    df_PML = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                                           intensity_image=img_PML_norm,
                                                           properties=['label', 'area', 'intensity_mean'],
                                                           spacing=voxel_size))
    df_PML['PML_sum_int_bgcorr'] = df_PML['area']/voxel_vol * df_PML['intensity_mean']
    
    df_SUMO = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                                           intensity_image=img_SUMO_norm,
                                                           properties=['intensity_mean'],
                                                           spacing=voxel_size))
    df_SUMO['SUMO_sum_int_bgcorr'] = df_PML['area']/voxel_vol * df_SUMO['intensity_mean']
    # Measure the Maximum intensity on Segmented DAPI channel to retrieve foci cell label 
  
    df_DAPI = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                               intensity_image=nucleus_focus,
                                               properties=['intensity_max'],
                                               spacing=voxel_size))

    # Add/Rename some columns
    df_PML = df_PML.rename(columns={'label': 'Foci_label', 'area': "Foci_vol_um³", 'intensity_mean': 'PML_int_mean_bgcorr'})
    df_SUMO = df_SUMO.rename(columns = {'intensity_mean' : 'SUMO_int_mean_bgcorr'})
    df_DAPI = df_DAPI.rename(columns = { 'intensity_max' : 'Nucleus_label'})
    # Concatenate the three table to get a single one
    df_comp = pd.concat([df_PML, df_SUMO, df_DAPI], axis=1)
    # Insert the image name at the beginning of every lines
    df_comp.insert(0, 'image', img_name)
    df_img_list.append(df_comp)

    # Create a mask without the detected focis, to measure intensity in cells but outside focis
    masked = filtered_label_image > 0
    masked_image = masked * nucleus_focus
    mask_xor = np.logical_xor(masked, masked_nucleus)
    xor_chanSUMO = mask_xor * img_SUMO_norm
    xor_masked_nucleus = mask_xor * masked_nucleus

    # New Measurements to get the intensity outside focis in the SUMO channel
    df_outside = pd.DataFrame(sk.measure.regionprops_table(xor_masked_nucleus, 
                                                   intensity_image=xor_chanSUMO,
                                                   properties=['label','area','intensity_mean'],
                                                   spacing=voxel_size))
    df_outside['Cell_int_sum_bgcorr'] = df_outside['area']/voxel_vol * df_outside['intensity_mean']
    df_img_outside = df_outside.rename(columns={'label': 'Nucleus_label', 'area': "Nucleus_vol_um³", 'intensity_mean': 'Nucleus_int_mean_bgcorr'})

    
    df_img_outside = df_img_outside.loc[:, ~df_img_outside.columns.duplicated()]
    df_img_outside.insert(0, 'image', img_name)
    df_mask_list.append(df_img_outside)

    # Create a last table with the backgrounds intensity for the 2 channels
    df_bg = pd.DataFrame({"PML_bg_int_mean": backgrounds_PML})
    df_bg.insert(0, "slice", range(1, len(backgrounds_PML)+1))
    df_bg.insert(0, 'image', img_name)
    df_bg.insert(2, "SUMO_bg_int_mean", backgrounds_SUMO)
    df_bg_list.append(df_bg)

# Concatenate the tables from all the images together and save them as csv    
df_img = pd.concat(df_img_list, ignore_index=True)
df_img = df_img.loc[:, ~df_img.columns.duplicated()]
df_img.to_csv(result_dir+'results.csv', index=False, float_format='%.3f')

df_mask = pd.concat(df_mask_list, ignore_index=True)
df_mask = df_mask.loc[:, ~df_mask.columns.duplicated()]
df_mask.to_csv(result_dir+'results_outside.csv', index=False, float_format='%.3f')

df_bg_final = pd.concat(df_bg_list, ignore_index=True)
df_bg_final = df_bg_final.loc[:, ~df_bg_final.columns.duplicated()]
df_bg_final.to_csv(result_dir+'backgrounds.csv', index=False, float_format='%.3f')

print("Analysis Done !")
print(f"Execution time: {datetime.datetime.now() - start_time}")


Analyzing image: C217Vn2 Mp2 1h DA-63x-1.czi (1/12)
[ 8  9 10 11 12 13]
Cells number: 19

Analyzing image: C217Vn2 Mp2 1h DA-63x-2.czi (2/12)
[ 8  9 10 11 12 13]
Cells number: 18

Analyzing image: C217Vn2 Mp2 2h DA-63x-1.czi (3/12)
[ 7  8  9 10 11]
Cells number: 23

Analyzing image: C217Vn2 Mp2 UT DA-63x-1.czi (4/12)
[ 8  9 10 11 12]
Cells number: 12

Analyzing image: E167R2 1h 63x-1.czi (5/12)
[ 9 10 11 12 13 14]
Cells number: 21

Analyzing image: E167R2 2h 63x-1.czi (6/12)
[ 9 10 11 12 13 14]
Cells number: 27

Analyzing image: E167R2 UT 63x-1.czi (7/12)
[ 8  9 10 11 12 13 14]
Cells number: 21

Analyzing image: E167R2 UT 63x-2.czi (8/12)
[ 8  9 10 11 12 13 14 15]
Cells number: 13

Analyzing image: WT 1h 63x-2.czi (9/12)
[10 11 12 13 14]
Cells number: 10

Analyzing image: WT 1h 63x-3.czi (10/12)
[10 11 12 13 14]
Cells number: 11

Analyzing image: WT 2h 63x-1.czi (11/12)
[10 11 12 13 14]
Cells number: 10

Analyzing image: WT UT 63x-1.czi (12/12)
[ 8  9 10]
Cells number: 9
Analysis Done

### 6. For 2D analysis 

In [11]:
start_time = datetime.datetime.now()
df_img_list = []
df_mask_list = []
df_bg_list = []
for index, img_name in enumerate(img_names, start=1):
    print(f"\nAnalyzing image: {img_name} ({index}/{len(img_names)})")
    img, voxel_size = open_img_czi(img_dir, img_name)
    

    # Select the Dapi channel and find best Z planes
    img_DAPI = img[channel_DAPI-1]

    # Open corresponding nucleus segmentation mask
    nucleus = sk.io.imread(result_dir + img_name.replace(".czi", "_nucleus.tif"))
  
    print(f"Cells number: {np.max(nucleus)}")

    
    # Keep nucleus segmentation on detected best z planes
    mask = nucleus > 0
    masked_nucleus = mask * nucleus 
    # Calculate the voxel volume wich is the product between every pixel size
    voxel_size = voxel_size[1]*voxel_size[2]
 
    # Took only best z planes on PML and SUMO channel
    chanPML = img[channel_PML-1,:,:]
    chanSUMO = img[channel_SUMO-1,:,:]
    
    # PREPROCESSING
    # Apply Difference of Gaussian filter to enhance Foci detection
    chanPML_dog = sk.filters.difference_of_gaussians(chanPML,1,3)  
    chanPML_dog = chanPML_dog[0,:,:]
    chanPML_dog_int = sk.util.img_as_int(chanPML_dog)
    
    # THRESHOLDING
    # Threshold the PML channel preprocessed with the triangle method
    thr = sk.filters.threshold_triangle(chanPML_dog_int)
    thr_PML_chan = chanPML_dog_int > thr
    
    # POSTPROCESSING
    # Perform a 3D opening to remove noise and to separate touching foci
    thr_PML_chan = sk.morphology.binary_opening(thr_PML_chan,footprint= sk.morphology.disk(2))

    # Put a label on every objects detected and filter them by volume
    chanPML_labels = sk.morphology.label(thr_PML_chan, connectivity=1)
    filter_out = sk.measure.regionprops(chanPML_labels)
    labels_to_keep = [obj.label for obj in filter_out]
    filtered_label_image = sk.morphology.label(np.isin(chanPML_labels, labels_to_keep))
    filtered_label_image = filter_labels_volume(filtered_label_image,
                                        min_voxels=foci_vol_min_um / voxel_size,
                                        max_voxels=foci_vol_max_um / voxel_size)
    
    
    # Compute the PML and SUMO backgrounds for every best z slice and substract it to the original image
    backgrounds_PML = compute_background_per_slice(chanPML)
    img_PML_norm = np.array([chanPML[i] - backgrounds_PML[i] for i in range(chanPML.shape[0])])
    backgrounds_SUMO = compute_background_per_slice(chanSUMO)
    img_SUMO_norm = np.array([chanSUMO[i] - backgrounds_SUMO[i] for i in range(chanSUMO.shape[0])])
    img_PML_norm = img_PML_norm[0,:,:]
    img_SUMO_norm = img_SUMO_norm[0,:,:]
    
    tf.imwrite(result_dir + img_name.replace(".czi", "_Foci_labels.tif"), filtered_label_image, imagej=False, resolution=(1/voxel_size, 1/voxel_size), metadata={'spacing': voxel_size, 'unit': 'um'})
    # MEASUREMENTS
    # Measure objects intensity in normalized channels (PML and SUMO) 
    df_PML = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                                           intensity_image=img_PML_norm,
                                                           properties=['label', 'area', 'intensity_mean'],
                                                           spacing=voxel_size))
    df_PML['PML_sum_int_bgcorr'] = df_PML['area']/voxel_size * df_PML['intensity_mean']
    
    df_SUMO = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                                           intensity_image=img_SUMO_norm,
                                                           properties=['intensity_mean'],
                                                           spacing=voxel_size))
    df_SUMO['SUMO_sum_int_bgcorr'] = df_PML['area']/voxel_size * df_SUMO['intensity_mean']
    # Measure the Maximum intensity on Segmented DAPI channel to retrieve foci cell label 
    df_DAPI = pd.DataFrame(sk.measure.regionprops_table(filtered_label_image, 
                                               intensity_image=nucleus,
                                               properties=['intensity_max'],
                                               spacing=voxel_size))

    # Add/Rename some columns
    df_PML = df_PML.rename(columns={'label': 'Foci_label', 'area': "Foci_vol_um³", 'intensity_mean': 'PML_int_mean_bgcorr'})
    df_SUMO = df_SUMO.rename(columns = {'intensity_mean' : 'SUMO_int_mean_bgcorr'})
    df_DAPI = df_DAPI.rename(columns = { 'intensity_max' : 'Nucleus_label'})
    # Concatenate the three table to get a single one
    df_comp = pd.concat([df_PML, df_SUMO, df_DAPI], axis=1)
    # Insert the image name at the beginning of every lines
    df_comp.insert(0, 'image', img_name)
    df_img_list.append(df_comp)

    # Create a mask without the detected focis, to measure intensity in cells but outside focis
    masked = filtered_label_image > 0
    masked_image = masked * nucleus
    mask_xor = np.logical_xor(masked, masked_nucleus)
    xor_chanSUMO = mask_xor * img_SUMO_norm
    xor_masked_nucleus = mask_xor * masked_nucleus

    # New Measurements to get the intensity outside focis in the SUMO channel
    df_outside = pd.DataFrame(sk.measure.regionprops_table(xor_masked_nucleus, 
                                                   intensity_image=xor_chanSUMO,
                                                   properties=['label','area','intensity_mean'],
                                                   spacing=voxel_size))
    df_outside['Nucleus_int_sum_bgcorr'] = df_outside['area']/voxel_size * df_outside['intensity_mean']
    df_img_outside = df_outside.rename(columns={'label': 'Nucleus_label', 'area': "Nucleus_vol_um³", 'intensity_mean': 'Nucleus_int_mean_bgcorr'})

    
    df_img_outside = df_img_outside.loc[:, ~df_img_outside.columns.duplicated()]
    df_img_outside.insert(0, 'image', img_name)
    df_mask_list.append(df_img_outside)

    # Create a last table with the backgrounds intensity for the 2 channels
    df_bg = pd.DataFrame({"PML_bg_int_mean": backgrounds_PML})
    df_bg.insert(0, "slice", range(1, len(backgrounds_PML)+1))
    df_bg.insert(0, 'image', img_name)
    df_bg.insert(2, "SUMO_bg_int_mean", backgrounds_SUMO)
    df_bg_list.append(df_bg)

# Concatenate the tables from all the images together and save them as csv    
df_img = pd.concat(df_img_list, ignore_index=True)
df_img = df_img.loc[:, ~df_img.columns.duplicated()]
df_img.to_csv(result_dir+'results.csv', index=False, float_format='%.3f')

df_mask = pd.concat(df_mask_list, ignore_index=True)
df_mask = df_mask.loc[:, ~df_mask.columns.duplicated()]
df_mask.to_csv(result_dir+'results_outside.csv', index=False, float_format='%.3f')

df_bg_final = pd.concat(df_bg_list, ignore_index=True)
df_bg_final = df_bg_final.loc[:, ~df_bg_final.columns.duplicated()]
df_bg_final.to_csv(result_dir+'backgrounds.csv', index=False, float_format='%.3f')

print("Analysis Done !")
print(f"Execution time: {datetime.datetime.now() - start_time}")


Analyzing image: NT 3pml 1 5000-1 PML488 SUMO2 594.czi (1/3)
Cells number: 31

Analyzing image: da1h pml 1 5000-1 PML488 SUMO2_594.czi (2/3)
Cells number: 23

Analyzing image: da2h pml 1 5000-3 PML_488 SUMO2_594.czi (3/3)
Cells number: 12
Analysis Done !


### Check cell compartments for the image of your choice

In [57]:
# Open image
img_name = img_names[-1]
print('Image name:', img_name)
img, voxel_size = open_img_czi(img_dir, img_name)


print(img.shape)
# Open corresponding segmentation masks for the inner and outer cell shells
Nucleus = sk.io.imread(result_dir + img_name.replace(".czi", "_Nucleus.tif"))
Focis = sk.io.imread(result_dir + img_name.replace(".czi", "_Foci_labels.tif"))
print(f"Cells number: {np.max(Nucleus)}")
        
# Display image and inner and outer cell shell mask overlay in Napari
viewer = get_napari_viewer()
viewer.add_image(img, name="image")
viewer.add_labels(Nucleus, opacity=1)
viewer.add_labels(Focis, opacity=0.5)

Image name: WT UT 63x-1.czi
(3, 20, 1024, 1024)
Cells number: 9


<Labels layer 'Focis' at 0x1ce14f65360>

### Visualize cell Aggrecan intensity for the image of your choice

### Check results file

In [41]:
df = pd.read_csv(result_dir+'results.csv')
df.head()

,image,Foci_label,Foci_vol_um³,PML_int_mean_bgcorr,PML_sum_int_bgcorr,SUMO_int_mean_bgcorr,SUMO_sum_int_bgcorr,Nucleus_label
0,C217Vn2 Mp2 1h DA-63x-1.czi,1,0.310,4780.538,1955240.081,116.849,47791.380,3.0
1,C217Vn2 Mp2 1h DA-63x-1.czi,2,0.297,3138.467,1233417.451,2833.598,1113603.925,5.0
2,C217Vn2 Mp2 1h DA-63x-1.czi,3,0.226,2438.154,729008.126,1593.260,476384.773,12.0
3,C217Vn2 Mp2 1h DA-63x-1.czi,4,0.107,1967.973,279452.130,410.001,58220.206,3.0
4,C217Vn2 Mp2 1h DA-63x-1.czi,5,0.456,11449.466,6892578.305,464.433,279588.844,3.0


In [42]:
df.describe()

,Foci_label,Foci_vol_um³,PML_int_mean_bgcorr,PML_sum_int_bgcorr,SUMO_int_mean_bgcorr,SUMO_sum_int_bgcorr,Nucleus_label
count,4579.000000,4579.000000,4579.000000,4.579000e+03,4579.000000,4.579000e+03,4579.000000
mean,208.173619,0.219474,2690.962999,1.153161e+06,1966.672608,7.282458e+05,8.690325
std,133.227010,0.167136,3088.208343,2.286347e+06,3504.152318,1.824606e+06,6.189043
min,1.000000,0.017000,344.409000,1.136549e+04,-239.651000,-4.502397e+04,0.000000
25%,96.000000,0.082000,1074.201000,1.231456e+05,410.536500,6.325766e+04,4.000000
50%,195.000000,0.188000,1672.732000,3.974944e+05,927.979000,1.911679e+05,8.000000
75%,307.000000,0.303000,3139.978000,1.145555e+06,1887.933000,4.942255e+05,13.000000
max,562.000000,0.798000,32560.759000,2.986624e+07,55861.237000,2.092033e+07,27.000000
